# Coffee water stress

> Synthetic triage records for the first task: water stress and equipment faults on coffee farms

In [ ]:
#| default_exp coffee

In the field, the triage comes from code in the domain project: feature engineering and an ML model flag the nodes to watch or fix for one farm and one day. The SLM turns that triage into a WhatsApp message for the farmer, as the [scope](design/scope.ipynb) describes. This module generates synthetic triage records, so that we can build and evaluate the message side before the real triage exists. It generates them in code, without a language model, so every record is valid and a seed reproduces it.

In [ ]:
#| export
import random
from datetime import date, timedelta
from string import ascii_uppercase
from edgeslm.synth import *

In [ ]:
#| hide
from fastcore.test import *
from collections import Counter

## Dimensions

The task has three dimensions with three levels each. A tuple sets the main item of a record: the kind of problem, its spatial pattern and its severity. For `both`, the record holds at least one item of each kind.

In [ ]:
#| export
DIMS = dict(problem=['water stress', 'equipment fault', 'both'],
            pattern=['one node', 'cluster', 'widespread'],
            severity=['early', 'moderate', 'severe'])
SEVERITIES = DIMS['severity']

In [ ]:
test_eq(len(tuples(DIMS)), 27)

## Farm layout

A farm is a list of groups of nodes, each group sharing an irrigation line. The names follow one of three schemes, drawn at random: greenhouse benches (small farms, 4 to 16 nodes), blocks with drip lines, or numbered plots (12 to 80 nodes). Location names are what the farmer reads, so they never contain node IDs.

In [ ]:
#| export
def layout(rng:random.Random)->list[tuple]:
    "A random farm: (group name, node locations) pairs, in one of three naming schemes"
    scheme = rng.choice(['benches', 'blocks', 'plots'])
    if scheme == 'benches':
        return [(f'Bench {b}', [f'Bench {b}, plant {p}' for p in range(1, rng.randint(2, 4)+1)])
                for b in range(1, rng.randint(2, 4)+1)]
    groups = []
    for g in range(rng.randint(3, 8)):
        rows = sorted(rng.sample(range(1, 30), rng.randint(4, 10)))
        if scheme == 'blocks':
            L = ascii_uppercase[g]
            groups.append((f'Block {L}, drip line {L}{rng.randint(1, 4)}', [f'Block {L}, row {r}' for r in rows]))
        else: groups.append((f'Plot {g+1}', [f'Plot {g+1}, row {r}' for r in rows]))
    return groups

In [ ]:
layout(random.Random(2))[:2]

[('Bench 1', ['Bench 1, plant 1', 'Bench 1, plant 2']),
 ('Bench 2', ['Bench 2, plant 1', 'Bench 2, plant 2', 'Bench 2, plant 3'])]

## Causes and onset

Each item has a likely cause, which the message turns into an action. The causes depend on the kind of problem and its spatial pattern. For equipment faults they also depend on whether data is lost: an early fault means readings that are intermittent or implausible, and moderate or severe faults mean no readings at all. These lists are domain content, so they are among the first things for an expert to audit.

In [ ]:
#| export
STRESS_CAUSES = {
    'one node': ['clogged dripper', 'damaged drip tube near the plant', 'unknown'],
    'cluster': ['blocked or leaking irrigation line', 'irrigation valve not opening', 'low water pressure'],
    'widespread': ['hot dry weather', 'irrigation did not run', 'irrigation too short for the heat']}

FAULT_CAUSES = {  # (pattern, data lost) -> causes
    ('one node', False): ['soil sensor gives implausible values', 'thermal camera images are noisy', 'readings arrive intermittently'],
    ('one node', True): ['soil sensor sends no readings', 'node battery empty', 'node sends no data'],
    ('cluster', False): ['readings from these nodes arrive intermittently'],
    ('cluster', True): ['no data from these nodes: radio link lost', 'no data from these nodes: power cut'],
    ('widespread', False): ['readings from most nodes arrive intermittently'],
    ('widespread', True): ['no data from most nodes: gateway radio problem', 'no data from most nodes: power cut']}

Severity also sets how long ago the problem started. `ONSET` gives the range of days for each kind and severity. For faults it follows the scope's definition: no readings for 1 to 3 days is moderate, and more than 3 days is severe.

In [ ]:
#| export
ONSET = {('water stress', 'early'): (0, 1), ('water stress', 'moderate'): (2, 4), ('water stress', 'severe'): (4, 8),
         ('equipment fault', 'early'): (0, 1), ('equipment fault', 'moderate'): (1, 3), ('equipment fault', 'severe'): (4, 10)}

## Items

One item is one problem at one location. A `one node` item names a node, a `cluster` item names a group, and a `widespread` item covers the whole farm with most of its nodes flagged. `used` holds the locations already taken in the record, so that two items never point at the same group.

In [ ]:
#| export
def _free(groups, used): return [(g, ns) for g, ns in groups if g not in used and not used & set(ns)]

def _item(rng, problem, pattern, severity, groups, used, day):
    "One triage item; adds its location to `used`"
    if pattern == 'widespread':
        n = sum(len(ns) for _, ns in groups)
        loc, counts = 'whole farm', dict(nodes_flagged=rng.randint(int(0.7*n)+1, n), nodes_in_group=n)
    else:
        g, ns = rng.choice(_free(groups, used))
        if pattern == 'one node': loc, counts = rng.choice(ns), {}
        else: loc, counts = g, dict(nodes_flagged=rng.randint(min(2, len(ns)), len(ns)), nodes_in_group=len(ns))
        used.add(loc)
    if problem == 'water stress': cause = rng.choice(STRESS_CAUSES[pattern])
    else: cause = rng.choice(FAULT_CAUSES[pattern, severity != 'early'])
    lo, hi = ONSET[problem, severity]
    return dict(problem=problem, location=loc, pattern=pattern, severity=severity,
                since=str(day - timedelta(rng.randint(lo, hi))), likely_cause=cause, **counts)

## Triage records

`triage` is the generator for `sample`. It builds the main item from the tuple, then adds random variation: the farm and its layout, a day in the Brazilian dry season (May to September), the weather, and up to two extra items. Extra items are never more severe than the main one, and are single nodes or clusters. For `both`, the first extra item is of the other kind. The weather is hot and dry whenever an item blames it.

In [ ]:
#| export
FARMS = ['Fazenda Boa Vista', 'Sítio Santa Luzia', 'Fazenda Cachoeira', 'Sítio Bela Vista', 'Fazenda São João', 'Sítio Recanto']
WEATHER = ['hot and dry', 'mild', 'cool and cloudy', 'rain in the last two days']
KINDS = ['water stress', 'equipment fault']

In [ ]:
#| export
def triage(t:dict, # Tuple: `problem`, `pattern` and `severity` of the main item
           rng:random.Random
          )->dict:
    "A random triage record for one farm and one day, whose main item matches `t`"
    groups, used = layout(rng), set()
    day = date(2026, 5, 1) + timedelta(rng.randint(0, 152))
    both = t['problem'] == 'both'
    main = rng.choice(KINDS) if both else t['problem']
    items = [_item(rng, main, t['pattern'], t['severity'], groups, used, day)]
    for i in range(rng.choice([1, 1, 2] if both else [0, 0, 1, 1, 2])):
        kind = KINDS[1 - KINDS.index(main)] if both and i == 0 else rng.choice(KINDS) if both else main
        if t['pattern'] == 'widespread' and kind == main: continue  # the main item already covers the farm
        if not _free(groups, used): break
        sev = rng.choice(SEVERITIES[:SEVERITIES.index(t['severity'])+1])
        items.append(_item(rng, kind, rng.choice(['one node', 'cluster']), sev, groups, used, day))
    hot = any(it['likely_cause'] == 'hot dry weather' for it in items)
    weather = 'hot and dry' if hot else rng.choice(WEATHER)
    rain = rng.randint(1, 2) if weather.startswith('rain') else rng.randint(5, 60)
    return dict(farm=rng.choice(FARMS), date=str(day), nodes=sum(len(ns) for _, ns in groups),
                context=dict(weather=weather, last_rain=str(day - timedelta(rain))), items=items)

A record for a moderate cluster with both kinds of problem:

In [ ]:
triage(dict(problem='both', pattern='cluster', severity='moderate'), random.Random(3))

{'farm': 'Sítio Recanto',
 'date': '2026-08-30',
 'nodes': 13,
 'context': {'weather': 'mild', 'last_rain': '2026-08-11'},
 'items': [{'problem': 'water stress',
   'location': 'Bench 1',
   'pattern': 'cluster',
   'severity': 'moderate',
   'since': '2026-08-26',
   'likely_cause': 'irrigation valve not opening',
   'nodes_flagged': 3,
   'nodes_in_group': 4},
  {'problem': 'equipment fault',
   'location': 'Bench 4',
   'pattern': 'cluster',
   'severity': 'early',
   'since': '2026-08-29',
   'likely_cause': 'readings from these nodes arrive intermittently',
   'nodes_flagged': 4,
   'nodes_in_group': 4}]}

## A test set

The test set has 10 records per tuple, 270 in total:

In [ ]:
recs = sample(DIMS, triage, 270, seed=1)
test_eq(set(Counter(tuple(r['tuple'].values()) for r in recs).values()), {10})

Every record keeps the rules above. The main item matches its tuple, no item is more severe than the tuple, every location appears once, and `both` records hold both kinds:

In [ ]:
items = lambda r: r['input']['items']
sev = SEVERITIES.index
for r in recs:
    t, main = r['tuple'], items(r)[0]
    test_eq((main['pattern'], main['severity']), (t['pattern'], t['severity']))
    assert all(sev(it['severity']) <= sev(t['severity']) for it in items(r))
    test_eq(len({it['location'] for it in items(r)}), len(items(r)))
    if t['problem'] == 'both': test_eq({it['problem'] for it in items(r)}, set(KINDS))

Records have one to three items, and farms range from a few nodes to about 80:

In [ ]:
Counter(len(items(r)) for r in recs), min(r['input']['nodes'] for r in recs), max(r['input']['nodes'] for r in recs)

(Counter({1: 112, 2: 111, 3: 47}), 4, 71)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()